# **Appendix A: Introduction to PyTorch (Part 2)**

## **9. Optimizing training performance with GPUs**

### **9.1. PyTorch computations on GPU devices**

In PyTorch, a device is where computations occur, and data resides. The CPU and the GPU are examples of devices. A PyTorch tensor resides in a device, and its operations are executed on the same device.

Let’s see how this works in action.

In [1]:
import torch

print(torch.__version__)

2.10.0+cu128


We can double-check that our runtime indeed supports GPU computing via the following code:

In [4]:
print(torch.cuda.is_available())

True


In [11]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])

print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


We can now use the `.to()` method to transfer these tensors onto a GPU and perform the addition there abd vice versa:

In [16]:
tensor_1 = tensor_1.to("cuda:0")
tensor_2 = tensor_2.to("cuda:0")

print(tensor_1 + tensor_2)

tensor([5., 7., 9.], device='cuda:0')


Notice that the resulting tensor now includes the device information, `device='cuda:0'`, which means that the tensors reside on the first GPU. If our machine hosts multiple GPUs, we have the option to specify which GPU we’d like to transfer the tensors to. We can do this by indicating the device ID in the transfer command. For instance, we can use `.to("cuda:0")`, `.to("cuda:1")`, and so on.

However, it is important to note that all tensors must be on the same device. Otherwise, the computation will fail, as shown below, where one tensor resides on the CPU and the other on the GPU:

In [13]:
tensor_1 = tensor_1.to("cpu")
print(tensor_1 + tensor_2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

### **9.2. Single-GPU training**

Before we get to the training loop, let's define the necessary objects first:

In [17]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

In [19]:
from torch.utils.data import Dataset


class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [20]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=1,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=1
)

In [21]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

We can modify the training loop from *section 2.7*, A typical training loop, to run on a GPU. This requires only changing three lines of code, as shown in the code below.

In [22]:
import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

# This is more conservative than model.to("cuda") in avoid errors if GPU is not available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #NEW
model.to(device)

# Note that the book originally used the following line, but the "model =" is redundant
# model = model.to(device) # NEW

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 002/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 002/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 002/002 | Train/Val Loss: 0.00


In [23]:
def compute_accuracy(model, dataloader, device):

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        features, labels = features.to(device), labels.to(device) # New

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [24]:
compute_accuracy(model, train_loader, device=device)

1.0

In [25]:
compute_accuracy(model, test_loader, device=device)

1.0

### **9.3 Training with multiple GPUs**

**Distributed training** accelerates deep learning by dividing the training workload across multiple GPUs or machines, which is essential for scaling up large models and speeding up experimentation.

#### **How DistributedDataParallel (DDP) Works**

PyTorch’s DDP is a data-parallel strategy that operates through the following steps:

1. **Model Replication:** PyTorch spawns a separate process for each available GPU, and every GPU receives an identical copy of the model.
2. **Data Splitting:** A `DistributedSampler` divides the dataset into unique, non-overlapping minibatches, distributing a distinct batch to each GPU.
3. **Independent Passes:** Each GPU independently runs its own forward and backward passes on its unique data subset, generating its own unique set of gradients.
4. **Gradient Synchronization:** Before updating the weights, the gradients from all GPUs are averaged and synchronized (via an "All-Reduce" operation). This ensures every model copy updates identically and the weights never diverge.

<figure style='text-align: center'>
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/12.webp" width="600px">
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/13.webp" width="600px">
</figure>

#### **Key Benefits & Limitations**

* **Scaling Efficiency:** By processing data simultaneously, two GPUs can cut training time nearly in half (minus minor communication overhead). This speedup scales linearly with additional hardware (e.g., 8 GPUs can approach an $8\times$ speedup).
* **Environment Restriction:** DDP requires spawning separate processes with their own Python interpreters. Because of this multiprocessing requirement, **DDP does not work inside interactive environments like Jupyter Notebooks** and must be executed as a standalone Python script (`.py`).

Implementing PyTorch’s `DistributedDataParallel` (DDP) requires specific utilities to initialize the environment, distribute the workload, and clean up system resources. The three core DDP utilties are as follows:

* **`init_process_group`:** Called at the very beginning of the script. It establishes the communication network (the "process group") that connects all independent GPU processes together.
* **`DistributedSampler`:** Wrapped around your dataset to partition the data. It ensures that each independent process/GPU receives a completely unique, non-overlapping chunk of the dataset.
* **`destroy_process_group`:** Called at the very end of the script. It safely terminates the distributed environment and releases the allocated hardware resources back to the system.

Here is a high-level snippet to demonstrate how these utilies should be used for parallel processing:

```python
# 1. Start the distributed backend
init_process_group(backend="nccl")

# 2. Partition the data so GPUs don't duplicate work
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, sampler=sampler)

# 3. Wrap model in DDP for automatic gradient synchronization
model = DistributedDataParallel(model)

# --- Training Loop Happens Here ---

# 4. Clean up resources when finished
destroy_process_group()

```

### **NCCL backend for optimized GPU communication**

NCCL (pronounced "Nickel") stands for the *NVIDIA Collective Communications Library*

It is a standalone software library developed by NVIDIA that provides highly optimized communication routines specifically designed for multi-GPU and multi-node deep learning training.

When we train a deep learning model across multiple GPUs (such as with PyTorch's `DistributedDataParallel`), the GPUs must constantly talk to one another. Every single training iteration requires the GPUs to share, average, and synchronize their gradients before updating the model's weights.

Standard CPU-based networking protocols are too slow to handle this massive, high-frequency data transfer. NCCL solves this by bypassing the CPU as much as possible, allowing GPUs to communicate with one another at maximum hardware speed.

If we are running on NVIDIA hardware, omitting or misconfiguring NCCL will usually result in your training loop running drastically slower or bottlenecking your GPUs entirely.

#### **How It Works: Key Techniques**

* **Hardware-Aware Topology:** NCCL automatically detects how your GPUs are physically connected. If they are plugged into ultra-fast internal bridges like **NVLink** or **NVSwitch**, it routes data through those. If they are on different machines across a network, it utilizes specialized network technologies like **InfiniBand** or **RoCE (RDMA over Converged Ethernet)**.
* **Collective Communication Operations:** Instead of writing custom point-to-point communication code, deep learning frameworks call NCCL's built-in collective algorithms. The most famous is **All-Reduce**, which aggregates tensors from all GPUs, calculates the average, and redistributes the final result back to every GPU simultaneously.

#### **NCCL vs. Other Backends**

When initializing distributed training in PyTorch (`init_process_group`), we typically choose a backend. NCCL is universally preferred for NVIDIA hardware. But there are other libraries that suites specific OS and hardware:

| Backend | Primary Hardware | Best Used For |
| --- | --- | --- |
| **`nccl`** | **NVIDIA GPUs** | **Distributed GPU Training** (The industry standard; fastest speed by a wide margin). |
| **`gloo`** | CPUs (Linux/Windows/macOS) | CPU-based distributed training or basic prototyping. |
| **`mpi`** | High-Performance Computing (HPC) | Legacy clusters or specific enterprise supercomputers. |

See [ddp.py](../../scripts/ddp.py)